# 04 — Cross-Subject Machine Learning (LOSO-CV)

**Goal:** prove (or disprove) subject-independent, zero-calibration decoding across all 50 acute stroke patients.

**Validation scheme:** Leave-One-Subject-Out Cross-Validation. Each fold trains CSP+LDA and Riemannian-MDM strictly on 49 subjects' pooled trials and evaluates on the 40 held-out trials of the remaining subject. Trials are never pooled across subjects before the split (Section 8, "Data Leakage in CSP").

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # optional — falls back to matplotlib if unavailable

from src.data_loader import load_all_subjects
from src.pipelines import evaluate_loso

## 1. Load all subjects

This can take a while the first time (downloading + filtering 50 subjects). Consider restricting `subject_ids` to a small list while iterating on this notebook, then rerunning on the full cohort.

In [ ]:
# X, y, subjects = load_all_subjects()  # full 50-subject run
X, y, subjects = load_all_subjects(subject_ids=None)
print(f'X shape: {X.shape}')
print(f'{len(np.unique(subjects))} unique subjects, {len(y)} total trials')
print(f'Class balance: {np.bincount(y)}')

## 2. Run LOSO-CV across both benchmark pipelines

In [ ]:
results = evaluate_loso(X, y, subjects, save_best_to='../models/trained_loso_pipeline.joblib')

## 3. Aggregate metrics

In [ ]:
def summarize(name, acc_key, bal_key, auc_key):
    acc = np.array(results[acc_key])
    bal = np.array(results[bal_key])
    auc = np.array(results[auc_key])
    print(f'{name}')
    print(f'  Accuracy:          {acc.mean():.3f} +/- {acc.std():.3f}')
    print(f'  Balanced accuracy: {bal.mean():.3f} +/- {bal.std():.3f}')
    print(f'  ROC-AUC:           {np.nanmean(auc):.3f} +/- {np.nanstd(auc):.3f}')
    print()

summarize('Pipeline A — CSP + LDA', 'csp_lda_acc', 'csp_lda_bal_acc', 'csp_lda_auc')
summarize('Pipeline B — Riemannian MDM', 'riemann_acc', 'riemann_bal_acc', 'riemann_auc')

## 4. Per-subject accuracy spread (subject-independent generalization is uneven)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x_pos = np.arange(len(results['fold_subject']))
ax.bar(x_pos - 0.2, results['csp_lda_acc'], width=0.4, label='CSP+LDA')
ax.bar(x_pos + 0.2, results['riemann_acc'], width=0.4, label='Riemannian MDM')
ax.axhline(0.5, color='gray', linestyle='--', label='chance')
ax.set_xticks(x_pos)
ax.set_xticklabels(results['fold_subject'], rotation=90, fontsize=7)
ax.set_xlabel('held-out subject')
ax.set_ylabel('accuracy')
ax.set_title('LOSO-CV per-subject accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Pooled confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, key, title in [
    (axes[0], 'csp_lda_confusion', 'CSP + LDA'),
    (axes[1], 'riemann_confusion', 'Riemannian MDM'),
]:
    cm = results[key]
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, str(val), ha='center', va='center')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['left', 'right'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['left', 'right'])
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{title} — pooled 50-subject confusion matrix')
plt.tight_layout()
plt.show()

## Interpretation notes
- Compare both pipelines' mean accuracy **and** their per-subject variance — a pipeline that's more consistent across patients may be preferable clinically even at a similar mean.
- Expect accuracy well below healthy-cohort benchmarks; report it honestly against chance (50%) rather than against unrelated healthy datasets (Section 9, "Clinical Realism").
- The saved `models/trained_loso_pipeline.joblib` is fit on **all** subjects for the Script 05 demo only — never cite its training accuracy as a generalization estimate; use the fold results above.

## Next
Run **`scripts/05_Simulated_Rehab_Interface.py`** to replay a held-out subject's recording through the trained pipeline in the simulated closed-loop Pygame interface.